# 02. 음원 → .nam (전체 경로)

음원 파일과 세션 JSON을 Drive에 올리고 실행한다. 결과는 `runs/<날짜_곡명>/`.
`SEGMENTS`를 비우면 추천 구간을 쓴다. Gradio 앱으로도 같은 과정을 할 수 있다(맨 아래).

In [ ]:
# 공통 준비: Drive 연결, 저장소 받기, 패키지 설치
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
DRIVE = '/content/drive/MyDrive/song2nam'   # 데이터·실행 기록·체크포인트를 둘 Drive 폴더
REPO = '/content/song2nam'
BRANCH = 'main'                               # 작업 브랜치로 바꿔서 쓸 수 있음
os.makedirs(f'{DRIVE}/data', exist_ok=True)
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-b', BRANCH, 'https://github.com/gmeag01/song2nam.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
os.chdir(REPO)
# data/, runs/를 Drive에 연결해서 Colab 런타임이 끊겨도 남게 한다
for d in ['data', 'runs']:
    if os.path.islink(d):
        continue
    subprocess.run(['rm', '-rf', d])
    os.symlink(f'{DRIVE}/{d}', d)

In [ ]:
# Colab에는 torch/torchaudio가 이미 있으므로 나머지만 설치
!grep -vE '^(torch|torchaudio)$' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
SONG = 'data/songs/my_song.wav'          # 업로드한 음원
SESSION = 'data/songs/my_song.json'      # 세션 JSON (설계 문서 4.3 스키마), 없으면 None
SEGMENTS = '42:58.5,131.2:140'           # 시작:끝,... (초). 비우면 추천 구간
ENCODER = 'checkpoints/encoder_v1_best.pt' # 없으면 None
import os
ENCODER = ENCODER if ENCODER and os.path.exists(ENCODER) else None

In [ ]:
from ampcap import pipeline
args = [SONG, '--egdb', 'data/EGDB/audio_DI', '--nam-input', 'data/nam/v3_0_0.wav', '--steps', '800', '--epochs', '100']
if SESSION: args += ['--session', SESSION]
if SEGMENTS: args += ['--segments', SEGMENTS]
if ENCODER: args += ['--encoder', ENCODER]
pipeline.main(args)

## (b) 세션만 경로 (비교 기준)

In [ ]:
pipeline.main([SONG, '--egdb', 'data/EGDB/audio_DI', '--session', SESSION, '--session-only',
               '--run-dir', 'runs/session_only_' + os.path.basename(SONG).split('.')[0]])

## Gradio 앱 (공개 링크)

In [ ]:
import os
os.environ['ANTHROPIC_API_KEY'] = ''   # ②-0 LLM 조사를 쓰려면 키 입력 (또는 Colab 보안 비밀 사용)
!python app.py --share --encoder checkpoints/encoder_v1_best.pt